In [1]:
#==============================================

#Tarefa 1.1, Carregamento e exploração de df_members

#GameStore

#==============================================
import pandas as pd
import numpy as np
import re as re

In [2]:
#carregando os arquivos csv#

print("Carregando os arquivos da pasta 'data'...")
df_games = pd.read_csv(r"C:\GameStoreBrasil\data\games.csv")
df_orders = pd.read_csv(r"C:\GameStoreBrasil\data\orders.csv")
df_members = pd.read_csv(r"C:\GameStoreBrasil\data\members.csv")
print("Arquivos carregados com sucesso")

Carregando os arquivos da pasta 'data'...
Arquivos carregados com sucesso


In [3]:
''' 
Devemos tratar idade antes de tratar as horas para que o calculo da mediana não seja afetado. Se eu calcular depois de dropar as linhas que tem NaT em join_date, por exemplo, eu vou
dropar linhas e afetarei o calculo da mediana, entende?
'''
#Pegando a mediana da idade
#● A mediana de idade foi calculada sobre os dados originais, antes de qualquer filtro

median_age = df_members['age'].median()


In [4]:
#● Nenhum valor nulo remanescente em age, phone_number e promotion_id

df_members['age'] = df_members['age'].fillna(median_age)
df_members['phone_number'] = df_members['phone_number'].fillna('0')
df_orders['promotion_id'] = df_orders['promotion_id'].fillna('0')

In [5]:
rng = np.random.default_rng(42)
def LimparNumeroTelefone(p):
    if p in (None, '0') or pd.isna(p):
        return p
    return re.sub(r'[^\d+]',"", str(p))

def AdicionarHorasEMinutosAleatorios(data_serie):
    data = pd.to_datetime(data_serie, errors = 'coerce')
    horas = rng.integers(9, 18, len(data))
    minutos = rng.integers(0,60, len(data))
    return data + pd.to_timedelta(horas, unit= 'h') + pd.to_timedelta(minutos, unit='m')


In [6]:
#● Telefones mantêm o caractere '+' e não contêm espaços, parênteses ou traços

df_members["phone_number"] = df_members["phone_number"].apply(LimparNumeroTelefone)
df_orders["order_date"] = AdicionarHorasEMinutosAleatorios(
    df_orders["order_date"]
)

df_members["join_date"] = AdicionarHorasEMinutosAleatorios(
    df_members["join_date"]
)

In [ ]:
#● Coluna de data é do tipo datetime64 (verificado com .dtypes) em ambos os arquivos

print(f"o tipo de dado da coluna phone_number é {df_members['phone_number'].dtype}, o da coluna order_date é {df_orders['order_date'].dtype} e o da join_date é {df_members['join_date'].dtype}")
print(f"df_members tem {df_members['phone_number'].isnull().sum()} valores nulos, df_orders tem {df_orders['promotion_id'].isnull().sum()} e df_members em join date tem {df_members['age'].isnull().sum()}")

o tipo de dado da coluna phone_number é str, o da coluna order_date é datetime64[us] e o da join_date é datetime64[us]
df_members tem 0 valores nulos, df_orders tem 0 e df_members em join date tem 0


In [8]:
#Nenhum horário está fixo em 00:00:00 (verificado com .dt.time.value_counts())

qtdMeiaNoite = (df_orders[df_orders['order_date'].dt.hour == 0]['order_date'].dt.time.value_counts())

In [9]:
import os

os.makedirs(r"C:\GameStoreBrasil\Output", exist_ok=True)

df_members.to_csv("output/members_cleaned.csv", index=False, encoding= 'utf-8')
df_orders.to_csv("output/orders_cleaned.csv", index=False, encoding= 'utf-8')